# HypatiaX Notebook 03: Hybrid System Experiments

**Paper:** LLMs as Interfaces to Symbolic Discovery: Perfect Extrapolation via Hybrid Architectures  
**Journal:** Journal of Machine Learning Research (2026)  
**Author:** Dr. Ruperto Pedro Bonet Chaple

This notebook documents the Hybrid v40 system results — the main contribution of the paper combining LLM with Neural Network fallback.

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# ── Repo-root resolution ─────────────────────────────────────────────────────
# Works whether the notebook is run from the repo root, the notebooks/ dir,
# or any subdirectory.  Looks for the marker file hypatiax/data/results.
def _find_repo_root() -> Path:
    """Walk up from this notebook until we find hypatiax/data/results."""
    here = Path().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'results').exists():
            return candidate
        if (candidate / 'hypatiax' / 'data' / 'results').exists():
            return candidate / 'hypatiax'
    raise FileNotFoundError(
        "Cannot locate repo root.  "
        "Run the notebook from inside the LLM-HypatiaX-PAPERS repository."
    )

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'data' / 'results'
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so hypatiax modules are importable
if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

print(f"✓ Repo root   : {REPO_ROOT}")
print(f"✓ Results dir : {RESULTS_DIR}")
print(f"✓ Figures dir : {FIGURES_DIR}")

MAIN_FILE    = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_131438.json'
MAIN_FILE2   = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_133510.json'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-paper')
sns.set_palette('husl')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})


with open(MAIN_FILE) as f:
    raw = json.load(f)

df = pd.DataFrame(raw)

# Extract all fields
df['equation_name'] = df['metadata'].apply(lambda x: x.get('equation_name', ''))
df['difficulty']    = df['metadata'].apply(lambda x: x.get('difficulty', ''))
df['formula_type']  = df['metadata'].apply(lambda x: x.get('formula_type', ''))
df['ground_truth']  = df['metadata'].apply(lambda x: x.get('ground_truth', ''))
df['llm_r2']        = df['llm_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['nn_r2']         = df['nn_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['eval_r2']       = df['evaluation'].apply(lambda x: x.get('r2', None) if x else None)
df['eval_rmse']     = df['evaluation'].apply(lambda x: x.get('rmse', None) if x else None)
df['success']       = df['evaluation'].apply(lambda x: x.get('success', False) if x else False)

print(f'✓ Loaded {len(df)} equations')

## 2. Hybrid System Overall Performance

In [ ]:
# Filter valid results
valid = df[df['eval_r2'].notna()].copy()
successful = valid[valid['eval_r2'] >= 0.90]

print('=== HYBRID SYSTEM RESULTS ===')
print(f'Total equations:      {len(df)}')
print(f'Valid results:        {len(valid)}')
print(f'Successful (R²≥0.90): {len(successful)} ({len(successful)/len(valid)*100:.1f}%)')
print(f'Failed (R²<0.90):     {len(valid)-len(successful)}')
print(f'\nMean R²:              {valid["eval_r2"].mean():.4f}')
print(f'Median R²:            {valid["eval_r2"].median():.4f}')
print(f'\nDecision routing:')
print(df['decision'].value_counts().to_string())

## 3. Performance by Domain

In [ ]:
domain_stats = valid.groupby('domain').agg(
    count=('eval_r2', 'count'),
    mean_r2=('eval_r2', 'mean'),
    median_r2=('eval_r2', 'median'),
    successful=('eval_r2', lambda x: (x >= 0.90).sum())
).round(4)
domain_stats['success_rate_%'] = (domain_stats['successful'] / domain_stats['count'] * 100).round(1)

print('Hybrid Performance by Domain:')
display(domain_stats)

## 4. Performance by Difficulty

In [ ]:
diff_stats = valid.groupby('difficulty').agg(
    count=('eval_r2', 'count'),
    mean_r2=('eval_r2', 'mean'),
    successful=('eval_r2', lambda x: (x >= 0.90).sum())
).round(4)
diff_stats['success_rate_%'] = (diff_stats['successful'] / diff_stats['count'] * 100).round(1)

print('Hybrid Performance by Difficulty:')
display(diff_stats)

## 5. LLM vs NN Decision Analysis

In [ ]:
# Compare LLM-routed vs NN-routed performance
llm_routed = valid[valid['decision'] == 'llm']
nn_routed  = valid[valid['decision'] == 'nn']

print('=== ROUTING ANALYSIS ===')
print(f'LLM-routed cases: {len(llm_routed)}')
print(f'  Mean R²:        {llm_routed["eval_r2"].mean():.4f}')
print(f'  Success rate:   {(llm_routed["eval_r2"] >= 0.90).mean()*100:.1f}%')
print()
print(f'NN-routed cases:  {len(nn_routed)}')
print(f'  Mean R²:        {nn_routed["eval_r2"].mean():.4f}')
print(f'  Success rate:   {(nn_routed["eval_r2"] >= 0.90).mean()*100:.1f}%')

## 6. Complete Results Table

In [ ]:
results_table = valid[['equation_name', 'domain', 'difficulty', 'decision',
                        'eval_r2', 'eval_rmse', 'success']].copy()
results_table['status'] = results_table['eval_r2'].apply(
    lambda r: '✓' if r >= 0.90 else '✗'
)
results_table = results_table.sort_values(['domain', 'eval_r2'], ascending=[True, False])

pd.set_option('display.max_rows', 50)
display(results_table.reset_index(drop=True))

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) R² by domain
ax = axes[0, 0]
domains = domain_stats.index.tolist()
colors = sns.color_palette('husl', len(domains))
bars = ax.bar(domains, domain_stats['success_rate_%'], color=colors, alpha=0.85)
ax.axhline(y=95.8, color='red', linestyle='--', linewidth=2, label='Paper: 95.8%')
ax.set_title('(a) Success Rate by Domain')
ax.set_ylabel('Success Rate (%)')
ax.set_ylim([0, 115])
ax.tick_params(axis='x', rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

# (b) R² distribution
ax = axes[0, 1]
ax.hist(valid['eval_r2'].clip(-1, 1), bins=20, color='#2ecc71', alpha=0.8, edgecolor='white')
ax.axvline(x=0.90, color='red', linestyle='--', linewidth=2, label='Threshold (0.90)')
ax.set_xlabel('Evaluation R²')
ax.set_ylabel('Frequency')
ax.set_title('(b) R² Score Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# (c) LLM vs NN R² comparison
ax = axes[1, 0]
ax.scatter(valid[valid['decision']=='llm']['llm_r2'].clip(-1,1),
           valid[valid['decision']=='llm']['eval_r2'].clip(-1,1),
           alpha=0.7, color='#3498db', label='LLM-routed', s=60)
ax.scatter(valid[valid['decision']=='nn']['nn_r2'].clip(-1,1),
           valid[valid['decision']=='nn']['eval_r2'].clip(-1,1),
           alpha=0.7, color='#e74c3c', label='NN-routed', s=60, marker='^')
ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Component R²')
ax.set_ylabel('Final Evaluation R²')
ax.set_title('(c) Component vs Final R²')
ax.legend()
ax.grid(True, alpha=0.3)

# (d) Success by difficulty
ax = axes[1, 1]
colors_diff = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c'}
bars = ax.bar(diff_stats.index, diff_stats['success_rate_%'],
              color=[colors_diff.get(d, '#3498db') for d in diff_stats.index], alpha=0.85)
ax.set_title('(d) Success Rate by Difficulty')
ax.set_ylabel('Success Rate (%)')
ax.set_ylim([0, 115])
ax.grid(True, alpha=0.3, axis='y')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', fontsize=9)

plt.suptitle('Hybrid System (v40) Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_hybrid_performance.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / '03_hybrid_performance.png', dpi=300, bbox_inches='tight')
print('✓ Saved hybrid performance figure')
plt.show()

---
**Previous:** [02_pure_llm_experiments.ipynb](02_pure_llm_experiments.ipynb)  
**Next:** [04_extrapolation_analysis.ipynb](04_extrapolation_analysis.ipynb)